# Planet Forest-Change Workflow: Part 2 of 3 · Data preparation (Planet sandbox via Sentinel Hub)
**You are in Part 2 of 3.** Part 1 found out *what* imagery exists; Part 2 searches the analysis-ready sandbox collection and **downloads one GeoTIFF per acquisition date**; Part 3 turns the annual composites into NDVI and change maps.
Conventions (numbering, band order, credentials, AOI) are listed at the top of Part 1.

## 2.1 Purpose and background

**Why this is a different service from Part 1.** The sandbox collection (`byoc-3f605f75-86c4-411a-b4ae-01c896f0e54e`, source `PS_ARD_SR_DAILY`, "analysis-ready PlanetScope surface reflectance") is hosted on **Sentinel Hub** (`services.sentinel-hub.com`), not on `api.planet.com`.
A Planet API key only works with `api.planet.com`; asking Sentinel Hub for the collection with that key returns **404**. This is why Part 2 authenticates with an **OAuth client ID + secret** instead.

**What Part 2 does**
1. Authenticates with the OAuth client (sections 2.4-2.5).
2. Searches the **Catalog API** for every date that has imagery over the AOI and saves that list (sections 2.6-2.7).
3. Downloads GeoTIFFs with the **Process API** (sections 2.9-2.10). There is no "asset activation" step as in the Data API: the Process API renders the pixels on request.
4. Explains how the per-date files become annual composites (section 2.11).

**Before running (one time)**
1. Open the Planet Account app (`insights.planet.com/account`) → **OAuth Clients** → **Create New**.
2. Copy the **client ID** and **client secret** immediately (the secret is shown only once).
3. The account needs Monitoring Credits (paid or trial). The sandbox data are licensed **CC-BY-NC** (non-commercial).

The collection covers **2021-01-01 to 2023-12-29** around (-6.77, -52.38). Every Process request consumes *Processing Units* from the account, which is why the notebook downloads sparingly (section 2.10).
The secret is never pasted into a cell; it is requested with `getpass`.

## 2.2 Environment

### 2.2.1 Install packages
The line `#%pip install requests pandas rasterio matplotlib numpy` is a comment (it starts with `#`), so it does not run; it lists what must be installed once.
`requests` = HTTP, `pandas` = tables, `numpy` = arrays, `rasterio` = reading GeoTIFFs (used in Part 3 and handy for checking downloads), `matplotlib` = plots. Uncomment the line only if one of them is missing.

In [42]:
#%pip install requests pandas rasterio matplotlib numpy

### 2.2.2 Imports
`json` (pretty-print API properties), `time` (token expiry and polite pauses), `Path` (file paths that work on macOS, Linux and Windows), `getpass` (hidden credential prompts), `numpy`, `pandas`, `requests`.
All imports are kept together in this one cell so the rest of the part reads without hidden dependencies.

In [43]:
import json
import time
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import requests

## 2.3 Settings
**Every constant that defines "what and where" lives in this one cell, so nothing is hidden inside the logic below.**

| Setting | Value | Why |
|---|---|---|
| `COLLECTION_ID` / `COLLECTION` | `3f605f75-…` → `byoc-3f605f75-…` | The sandbox collection ID from the Planet account; Sentinel Hub names user collections **BYOC** ("bring your own COG"), hence the `byoc-` prefix. |
| `BBOX` | `[-52.4825, -6.8735, -52.26725, -6.65775]` | West, south, east, north in EPSG:4326: the AOI of Part 1 as a rectangle (the Catalog and Process APIs accept a bounding box directly). |
| `START_DATE` / `END_DATE` | 2013-01-01 → 2026-01-31 | Deliberately **wider** than the data. The collection only holds 2021-01-01 → 2023-12-29, so the search returns exactly what exists without having to know the edge dates. |
| `WIDTH` / `HEIGHT` | 2400 × 2400 | The Process API allows at most 2500 × 2500 pixels. 2400 px across ≈ 24 km gives **≈ 10 m per pixel**, a balance between detail and request cost. The native resolution is ≈ 3 m; a smaller `BBOX` is needed to use it. |
| `BASE_URL` | `https://services.sentinel-hub.com` | The EU deployment, where the Planet sandbox normally lives. The comment lists the US-West deployment to try if authentication or search says the collection does not exist. |

In [44]:
COLLECTION_ID = "3f605f75-86c4-411a-b4ae-01c896f0e54e"
COLLECTION = f"byoc-{COLLECTION_ID}"

# Area of interest: [west, south, east, north] in EPSG:4326
BBOX = [-52.4825, -6.8735, -52.26725, -6.65775]

# Must fall inside 2021-01-01 .. 2023-12-29
START_DATE = "2013-01-01T00:00:00Z"
END_DATE = "2026-01-31T23:59:59Z"

# Output size in pixels (Process API maximum is 2500 x 2500).
# 2400 px over ~24 km is ~10 m per pixel. The native resolution is 3 m,
# so use a smaller BBOX if you want full resolution.
WIDTH = 2400
HEIGHT = 2400

# Sentinel Hub deployment. The Planet sandbox is normally on the EU one.
# If auth or search says the collection does not exist, try the US one:
#   "https://services-uswest2.sentinel-hub.com"
BASE_URL = "https://services.sentinel-hub.com"

print("Collection:", COLLECTION)
print("Date range:", START_DATE, "to", END_DATE)

Collection: byoc-3f605f75-86c4-411a-b4ae-01c896f0e54e
Date range: 2013-01-01T00:00:00Z to 2026-01-31T23:59:59Z


## 2.4 Credentials

### 2.4.1 Enter the OAuth client (private)
`getpass` hides what you type. `.strip()` removes accidental spaces or line breaks from copy-paste, a frequent cause of "invalid client" errors. The cell **refuses to continue if either value is empty** (clear error now rather than an HTTP 400 later) and prints only "Credentials loaded", never the values.

In [45]:
CLIENT_ID = getpass("Planet OAuth client ID: ").strip()
CLIENT_SECRET = getpass("Planet OAuth client secret: ").strip()

if not CLIENT_ID or not CLIENT_SECRET:
    raise RuntimeError("Client ID and secret are both required.")

print("Credentials loaded.")

Credentials loaded.


## 2.5 Authentication

### 2.5.1 Get and cache an access token
Sentinel Hub uses the OAuth **client-credentials** flow: the client ID and secret are exchanged for a short-lived *bearer token* that is sent with every request.

- `_token = {"value": None, "expires_at": 0.0}`: a small dictionary acts as a cache. A dictionary is mutable, so the function can update it without a `global` statement.
- `get_token()` reuses the token while it is still valid and asks for a new one **60 seconds before** the stated expiry, so a long download cannot fail halfway because the token ran out.
- On failure it prints the HTTP status and the first 1000 characters of the answer (enough to read the reason, short enough not to flood the screen), then `raise_for_status()` stops the run. The secret itself is never printed.
- `expires_in` falls back to 3600 s if the server does not send it.
- `auth_headers(**extra)` builds the `Authorization: Bearer …` header and merges extra headers (for example `Content-Type`), so every later call needs one line.
- The last two lines request a token immediately so a wrong secret is discovered **now**, not in the middle of the search.

In [46]:
_token = {"value": None, "expires_at": 0.0}


def get_token():
    """Return a valid bearer token, fetching a new one only when needed."""
    if _token["value"] and time.time() < _token["expires_at"] - 60:
        return _token["value"]

    r = requests.post(
        f"{BASE_URL}/auth/realms/main/protocol/openid-connect/token",
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
        },
        timeout=60,
    )

    if not r.ok:
        print("Token request failed:", r.status_code)
        print(r.text[:1000])
        r.raise_for_status()

    data = r.json()
    _token["value"] = data["access_token"]
    _token["expires_at"] = time.time() + int(data.get("expires_in", 3600))
    return _token["value"]


def auth_headers(**extra):
    return {"Authorization": f"Bearer {get_token()}", **extra}


get_token()
print("Authentication succeeded.")

Authentication succeeded.


## 2.6 Search the catalog

### 2.6.1 List every date with imagery over the AOI
The **Catalog API** answers "which scenes exist?" without returning pixels (so it costs no Processing Units).

- `catalog_search(body)` tries the two documented paths `/api/v1/catalog/1.0.0/search` and `/catalog/v1/search`. They are aliases; the second is only used if the first answers **404**, which keeps the notebook working on deployments that expose only one of them.
- `body`: `collections` (which collection), `bbox` (where), `datetime` (the wide range of section 2.3), `limit: 100` (page size).
- **Pagination:** the answer contains `context.next`; it is sent back as `body["next"]` to get the next page. The loop is capped at **20 pages** (up to 2000 scenes) so a faulty `next` token cannot loop forever; the catalog for this AOI holds 677 items.
- **Error messages by status code:** 401 = credentials rejected, 403 = no access / no Monitoring Credits, 404 = try the other deployment in section 2.3. Each hint tells the reader what to change.
- If nothing is found the cell prints a reminder of the collection's date range and the AOI.

In [47]:
def catalog_search(body):
    # Try the documented catalog paths in turn (they are aliases of each other).
    last = None
    for path in ("/api/v1/catalog/1.0.0/search", "/catalog/v1/search"):
        r = requests.post(
            BASE_URL + path,
            headers=auth_headers(**{"Content-Type": "application/json"}),
            json=body,
            timeout=120,
        )
        if r.status_code != 404:
            return r
        last = r
    return last


body = {
    "collections": [COLLECTION],
    "bbox": BBOX,
    "datetime": f"{START_DATE}/{END_DATE}",
    "limit": 100,
}

features = []

for page in range(20):
    r = catalog_search(body)

    if not r.ok:
        print("HTTP status:", r.status_code)
        print(r.text[:2000])
        if r.status_code == 401:
            print("-> Credentials rejected. Check client ID/secret.")
        elif r.status_code == 403:
            print("-> Account may lack access/Monitoring Credits for this collection.")
        elif r.status_code == 404:
            print("-> Try the other deployment (BASE_URL in Cell 3).")
        r.raise_for_status()

    result = r.json()
    features.extend(result.get("features", []))

    nxt = result.get("context", {}).get("next")
    if not nxt:
        break
    body["next"] = nxt

print("Returned features:", len(features))

if not features:
    print("No scenes found. The collection covers 2021-01-01 to 2023-12-29; "
          "check START_DATE/END_DATE and that BBOX is inside the sandbox area.")

Returned features: 677


## 2.7 Inspect and save the catalog results

### 2.7.1 Look at the returned properties
- The first feature's `properties` are printed as indented JSON (cut at 1500 characters) so you can **see which fields the catalog really offers** before relying on them.
- Each row keeps the `id`, the `datetime` and **every property whose name contains "cloud"** (`k.lower()` makes the match case-insensitive). The cloud field's exact name is not known in advance, so matching on the word keeps the notebook robust.
- `results_df["date"] = results_df["datetime"].str[:10]` keeps only `YYYY-MM-DD`, because downloads (section 2.10) are requested **per day**.
- The cell raises a clear `RuntimeError` if the search returned nothing, then shows the first 30 rows and the sorted list of distinct dates.

In [48]:
if not features:
    raise RuntimeError("No scenes found. See the message above.")

print("Properties available on the first feature:")
print(json.dumps(features[0].get("properties", {}), indent=2)[:1500])

rows = []
for f in features:
    p = f.get("properties", {})
    row = {"id": f.get("id"), "datetime": p.get("datetime")}
    for k, v in p.items():
        if "cloud" in k.lower():
            row[k] = v
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df["date"] = results_df["datetime"].str[:10]
display(results_df.head(30))

print("\nDates with data:", sorted(results_df["date"].unique()))

Properties available on the first feature:
{
  "datetime": "2023-12-29T00:00:00Z",
  "proj:epsg": 32722,
  "proj:bbox": [
    336000.0,
    9240000.0,
    360000.0,
    9264000.0
  ],
  "proj:geometry": {
    "type": "Polygon",
    "crs": {
      "type": "name",
      "properties": {
        "name": "urn:ogc:def:crs:EPSG::32722"
      }
    },
    "coordinates": [
      [
        [
          336000.00000084564,
          9240000.00000001
        ],
        [
          360000.00000031735,
          9240000.000000002
        ],
        [
          360000.0000003204,
          9264000.000000002
        ],
        [
          336000.00000085135,
          9264000.00000001
        ],
        [
          336000.00000084564,
          9240000.00000001
        ]
      ]
    ]
  }
}


,id,datetime,date
0,4fd3e6eb-1f6d-458a-bbae-79d66b583302,2023-12-29T00:00:00Z,2023-12-29
1,0792c687-c0f3-4c96-8080-1394cafe91ad,2023-12-28T00:00:00Z,2023-12-28
2,84298261-298a-46be-8ca3-b7ca814a48ea,2023-12-25T00:00:00Z,2023-12-25
3,b5658ba1-25c9-4f26-b057-ad096dab02a7,2023-12-20T00:00:00Z,2023-12-20
4,c521fc07-d467-486d-b95b-584856955cf1,2023-12-19T00:00:00Z,2023-12-19
5,4f92ec3f-df19-4c4d-a42d-d82aab42297c,2023-12-16T00:00:00Z,2023-12-16
6,e74712b2-5424-4115-b785-0bbb1833ec4b,2023-12-15T00:00:00Z,2023-12-15
7,4953a8f0-d1f7-4bd5-9b3d-d95d43ae2de5,2023-12-14T00:00:00Z,2023-12-14
8,8663ed59-1b2a-4c47-a181-49274d9fd961,2023-12-13T00:00:00Z,2023-12-13
9,48e44399-d92a-41c2-af78-e479a55a76bd,2023-12-08T00:00:00Z,2023-12-08



Dates with data: ['2021-01-01', '2021-01-02', '2021-01-05', '2021-01-06', '2021-01-07', '2021-01-08', '2021-01-18', '2021-01-22', '2021-01-26', '2021-01-27', '2021-02-01', '2021-02-02', '2021-02-03', '2021-02-07', '2021-02-24', '2021-03-05', '2021-03-15', '2021-03-16', '2021-03-20', '2021-03-21', '2021-03-23', '2021-03-24', '2021-03-26', '2021-03-27', '2021-03-29', '2021-03-30', '2021-04-03', '2021-04-04', '2021-04-07', '2021-04-08', '2021-04-09', '2021-04-11', '2021-04-12', '2021-04-13', '2021-04-14', '2021-04-15', '2021-04-16', '2021-04-24', '2021-04-25', '2021-04-26', '2021-04-29', '2021-05-02', '2021-05-03', '2021-05-04', '2021-05-06', '2021-05-08', '2021-05-10', '2021-05-11', '2021-05-12', '2021-05-13', '2021-05-15', '2021-05-16', '2021-05-17', '2021-05-20', '2021-05-21', '2021-05-22', '2021-05-23', '2021-05-24', '2021-05-25', '2021-05-26', '2021-05-27', '2021-05-28', '2021-05-29', '2021-05-30', '2021-05-31', '2021-06-01', '2021-06-02', '2021-06-03', '2021-06-04', '2021-06-05', '

### 2.7.2 Save the catalog (`all_images_sentinel.csv`)
One line writes `results_df` to disk without the index. This file is a snapshot of **everything the catalog lists for the AOI (677 items, 2021-01-01 → 2023-12-29)**, i.e. the images that are *available*, whether or not they were downloaded. The Streamlit app reads it to show "images in catalog" next to "images downloaded".

In [49]:
results_df.to_csv("all_images_sentinel.csv", index=False)

## 2.8 Choose a date (single test download)

### 2.8.1 Select the date to test with
- `SELECTED_DATE = None` means "choose for me"; type a date such as `"2022-01-14"` to override.
- **Automatic rule:** the date with the **lowest mean cloud value**. The mean is taken per date (`groupby("date")`) because a day can contain several scenes. If the catalog carries no cloud field, the earliest date is used.
- The cell prints which rule was applied and the chosen date, so the choice is traceable.
A single, low-cloud date is the cheapest way to prove that the download works before spending credits on many dates.

In [50]:
SELECTED_DATE = None  # e.g. "2022-01-14"

cloud_cols = [c for c in results_df.columns if "cloud" in c.lower()]

if SELECTED_DATE is None:
    if cloud_cols:
        per_date = results_df.groupby("date")[cloud_cols[0]].mean().sort_values()
        SELECTED_DATE = per_date.index[0]
        print(f"Lowest mean {cloud_cols[0]}:", round(float(per_date.iloc[0]), 2))
    else:
        SELECTED_DATE = sorted(results_df["date"].unique())[0]

print("Selected date:", SELECTED_DATE)

Selected date: 2021-01-01


## 2.9 Download one GeoTIFF with the Process API (reference version)

### 2.9.1 Try the band sets one after the other
**Important:** the whole cell below is wrapped in a triple-quoted string (`""" … """`). Python therefore reads it as text and **does not execute it**; it is kept as the documented, step-by-step version of the download. The version that actually runs and wrote the files on disk is section 2.10.

What the reference version does, and why:
- `CANDIDATE_BANDS` lists three band-name sets: the 4-band surface-reflectance set (`blue, green, red, nir`), the same plus `rededge`, and the 8-band set of the newer PlanetScope generation (`coastal_blue, blue, green_i, green, yellow, red, rededge, nir`). Band names differ by sensor generation, so it **tries the simplest first and falls back**. The first set that succeeds is stored in `used_bands`.
- `build_evalscript(bands)` writes the small JavaScript program (an **evalscript**) that Sentinel Hub runs per pixel: it requests the chosen bands **plus `dataMask`**, and returns them as `FLOAT32` so reflectance keeps its decimals.
  `dataMask` is added as the **last band** (1 = valid pixel, 0 = no data), which is why the files have 5 bands.
- `process_request(bands)` builds the request: the `bbox` and its CRS, the collection, a **time range covering only `SELECTED_DATE`** (00:00:00-23:59:59 Z), `mosaickingOrder: "mostRecent"` (if several scenes overlap on that day, the most recent one is on top), the output size from section 2.3 and `image/tiff` as the response format.
- `timeout=600` (10 min) because rendering a 2400 × 2400 image can take a while.
- It saves `planet_downloads/ARPS_Para_<date>.tif` (`ARPS` = analysis-ready PlanetScope; `Para` = region). A **401, 403 or 429** stops immediately (credentials, permission or rate limit cannot be fixed by trying another band set); any other error prints the first 600 characters, which usually names the invalid band, and the next set is tried. If all fail, a `RuntimeError` explains what to read.
- It ends by printing the file path, its size and the band list.

In [51]:
""" 
CANDIDATE_BANDS = [
    ["blue", "green", "red", "nir"],
    ["blue", "green", "red", "nir", "rededge"],
    ["coastal_blue", "blue", "green_i", "green", "yellow", "red", "rededge", "nir"],
]


def build_evalscript(bands):
    inputs = ", ".join(f'"{b}"' for b in bands + ["dataMask"])
    returns = ", ".join(f"s.{b}" for b in bands + ["dataMask"])
#    return f### //VERSION=3
function setup() {{
  return {{
    input: [{{ bands: [{inputs}] }}],
    output: {{ bands: {len(bands) + 1}, sampleType: "FLOAT32" }}
  }};
}}
function evaluatePixel(s) {{
  return [{returns}];
}}
###


def process_request(bands):
    payload = {
        "input": {
            "bounds": {
                "bbox": BBOX,
                "properties": {"crs": "http://www.opengis.net/def/crs/EPSG/0/4326"},
            },
            "data": [{
                "type": COLLECTION,
                "dataFilter": {
                    "timeRange": {
                        "from": f"{SELECTED_DATE}T00:00:00Z",
                        "to": f"{SELECTED_DATE}T23:59:59Z",
                    },
                    "mosaickingOrder": "mostRecent",
                },
            }],
        },
        "output": {
            "width": WIDTH,
            "height": HEIGHT,
            "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}],
        },
        "evalscript": build_evalscript(bands),
    }
    return requests.post(
        f"{BASE_URL}/api/v1/process",
        headers=auth_headers(**{"Content-Type": "application/json", "Accept": "image/tiff"}),
        json=payload,
        timeout=600,
    )


OUTPUT_DIR = Path("planet_downloads")
OUTPUT_DIR.mkdir(exist_ok=True)
output_file = OUTPUT_DIR / f"ARPS_Para_{SELECTED_DATE}.tif"

used_bands = None

for bands in CANDIDATE_BANDS:
    print("Trying bands:", bands)
    r = process_request(bands)
    print("  HTTP status:", r.status_code)

    if r.ok:
        output_file.write_bytes(r.content)
        used_bands = bands
        break

    print("  ", r.text[:600])
    if r.status_code in (401, 403, 429):
        r.raise_for_status()

if used_bands is None:
    raise RuntimeError(
        "All band sets failed. Read the error text above: it usually names "
        "the invalid band or the access problem. Set CANDIDATE_BANDS to the "
        "band names it reports."
    )

print("Saved:", output_file)
print("File size:", f"{output_file.stat().st_size:,}", "bytes")
print("Bands in file:", used_bands + ["dataMask"])

"""

' \nCANDIDATE_BANDS = [\n    ["blue", "green", "red", "nir"],\n    ["blue", "green", "red", "nir", "rededge"],\n    ["coastal_blue", "blue", "green_i", "green", "yellow", "red", "rededge", "nir"],\n]\n\n\ndef build_evalscript(bands):\n    inputs = ", ".join(f\'"{b}"\' for b in bands + ["dataMask"])\n    returns = ", ".join(f"s.{b}" for b in bands + ["dataMask"])\n#    return f### //VERSION=3\nfunction setup() {{\n  return {{\n    input: [{{ bands: [{inputs}] }}],\n    output: {{ bands: {len(bands) + 1}, sampleType: "FLOAT32" }}\n  }};\n}}\nfunction evaluatePixel(s) {{\n  return [{returns}];\n}}\n###\n\n\ndef process_request(bands):\n    payload = {\n        "input": {\n            "bounds": {\n                "bbox": BBOX,\n                "properties": {"crs": "http://www.opengis.net/def/crs/EPSG/0/4326"},\n            },\n            "data": [{\n                "type": COLLECTION,\n                "dataFilter": {\n                    "timeRange": {\n                        "from": f"

## 2.10 Download every 7th date (the production download)

### 2.10.1 Select dates, authenticate and download
This is the cell that creates the per-date GeoTIFFs used by the rest of the workbook. It is written as **one self-contained script** with seven numbered blocks, so it can be run alone after a kernel restart (only `results_df` from section 2.7 is required).

| Block | What it does | Decision / reason |
|---|---|---|
| 1. Settings | Re-declares collection, `BBOX`, `BBOX_CRS`, dates, `BASE_URL`, output folder and `used_bands = ["blue","green","red","nir"]`. | Self-contained on purpose. The 4-band set is the one that worked in section 2.9. `OUTPUT_DIR = Path.cwd() / "planet_downloads"` keeps downloads next to the notebook; `mkdir(parents=True, exist_ok=True)` makes the folder without error if it exists. |
| 2. Check catalog | Stops with a clear `NameError`/`ValueError` if `results_df` is missing or empty; builds `all_dates`; `picked = all_dates[::STEP]`; always appends the **last** available date. | `STEP = 7`: **take every 7th date** (about one acquisition per week-sample) to cap Processing-Unit use and time while still covering the whole year; the first date is included by the slice, the last is added explicitly so the series always spans the full period. **`STEP` is the one number to change** for a denser or sparser download. |
| 3. OAuth | Reuses `CLIENT_ID`/`CLIENT_SECRET` from section 2.4 if they exist, otherwise asks with `getpass`; defines `get_access_token()` and `auth_headers()`; tests the login immediately. | Same flow as section 2.5, repeated so the script runs alone. The token is renewed 60 s before expiry; the failure message includes status and the first 500 characters of the answer. |
| 4. Evalscript | `build_evalscript(bands)` as in section 2.9 (bands + `dataMask`, `FLOAT32`). | Same reasoning: keep decimals, and carry a validity band instead of guessing which pixels are empty. |
| 5. Request | `process_request(date_string)` for one day with `mosaickingOrder: "mostRecent"`. | One request per day keeps each file a clean snapshot of a single date. |
| 6. Download loop | Skips existing non-empty files; saves new ones; on 401/403/429 prints why and **stops**; on any other exception stops after printing; `time.sleep(1)` between requests. | **Resumable:** re-running after an interruption continues where it stopped. Stopping on authentication, permission or quota errors avoids burning credits on requests that cannot succeed. The 1 s pause keeps the request rate polite. |
| 7. Summary | Prints counts, band list, folder and any failed dates. | Lets you see at once whether the run was complete. |

**Two things to know about this cell**
- **Image size.** The cell sets `WIDTH = HEIGHT = 1024`, whereas section 2.3 uses 2400 and the files currently on disk are **2400 × 2400** (they were created with the larger size). Set both to 2400 here if you re-download, so that new files match the existing ones and the composites.
- **Number of files.** `STEP = 7` on the 677-item catalog would give far fewer files than the **317** now in `planet_downloads/` (113 for 2021, 108 for 2022, 96 for 2023). The folder was therefore evidently filled over several runs with a smaller step. The "skip existing files" rule makes that safe.

In [55]:
from pathlib import Path
from getpass import getpass
import time
import requests

# ============================================================
# 1. SETTINGS
# ============================================================

STEP = 7

COLLECTION_ID = "3f605f75-86c4-411a-b4ae-01c896f0e54e"
COLLECTION = f"byoc-{COLLECTION_ID}"

# Area of interest: west, south, east, north
# Coordinates are longitude/latitude in EPSG:4326
BBOX = [-52.4825, -6.8735, -52.26725, -6.65775]
BBOX_CRS = "http://www.opengis.net/def/crs/EPSG/0/4326"

START_DATE = "2013-01-01T00:00:00Z"
END_DATE = "2026-01-31T23:59:59Z"

BASE_URL = "https://services.sentinel-hub.com"

WIDTH = 1024
HEIGHT = 1024

OUTPUT_DIR = Path.cwd() / "planet_downloads"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use the band names that worked in your previous Cell 9.
# If your earlier test used a different list, replace this list.
used_bands = ["blue", "green", "red", "nir"]

# ============================================================
# 2. CHECK CATALOG RESULTS
# ============================================================

if "results_df" not in globals():
    raise NameError(
        "results_df is missing. Run your Catalog search cell first."
    )

if len(results_df) == 0:
    raise ValueError("results_df is empty. No catalog results were found.")

all_dates = sorted(
    results_df["date"].astype(str).unique()
)

if len(all_dates) == 0:
    raise ValueError("No dates were found in results_df.")

# Select every STEP-th date
picked = all_dates[::STEP]

# Always include the last available acquisition date
if all_dates[-1] not in picked:
    picked.append(all_dates[-1])

print("Collection:", COLLECTION)
print("BBOX:", BBOX)
print("BBOX CRS:", BBOX_CRS)
print("Bands:", used_bands)
print(f"{len(all_dates)} dates found")
print(f"{len(picked)} dates selected")
print("Output folder:", OUTPUT_DIR.resolve())

# ============================================================
# 3. OAUTH AUTHENTICATION
# ============================================================

if "CLIENT_ID" not in globals() or not CLIENT_ID:
    CLIENT_ID = getpass("Planet OAuth client ID: ").strip()

if "CLIENT_SECRET" not in globals() or not CLIENT_SECRET:
    CLIENT_SECRET = getpass("Planet OAuth client secret: ").strip()

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError(
        "CLIENT_ID and CLIENT_SECRET cannot be empty."
    )

TOKEN_URL = (
    f"{BASE_URL}/auth/realms/main/protocol/openid-connect/token"
)

_access_token = None
_token_expires_at = 0


def get_access_token():
    global _access_token, _token_expires_at

    response = requests.post(
        TOKEN_URL,
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
        },
        headers={
            "Content-Type": "application/x-www-form-urlencoded"
        },
        timeout=30,
    )

    if not response.ok:
        raise RuntimeError(
            "Token request failed\n"
            f"HTTP {response.status_code}\n"
            f"{response.text[:500]}"
        )

    token_data = response.json()

    _access_token = token_data["access_token"]

    expires_in = token_data.get("expires_in", 3600)
    _token_expires_at = time.time() + expires_in - 60

    return _access_token


def auth_headers():
    if (
        _access_token is None
        or time.time() >= _token_expires_at
    ):
        get_access_token()

    return {
        "Authorization": f"Bearer {_access_token}",
        "Content-Type": "application/json",
        "Accept": "image/tiff",
    }


# Test authentication now
get_access_token()
print("Authentication successful.")

# ============================================================
# 4. EVALSCRIPT
# ============================================================

def build_evalscript(bands):
    input_bands = list(bands) + ["dataMask"]

    input_text = ", ".join(
        f'"{band}"' for band in input_bands
    )

    output_text = ", ".join(
        f"sample.{band}" for band in input_bands
    )

    return f"""//VERSION=3

function setup() {{
  return {{
    input: [{{
      bands: [{input_text}]
    }}],
    output: {{
      bands: {len(input_bands)},
      sampleType: "FLOAT32"
    }}
  }};
}}

function evaluatePixel(sample) {{
  return [{output_text}];
}}
"""

# ============================================================
# 5. CREATE PROCESSING API REQUEST
# ============================================================

def process_request(date_string):
    payload = {
        "input": {
            "bounds": {
                "bbox": BBOX,
                "properties": {
                    "crs": BBOX_CRS
                }
            },
            "data": [
                {
                    "type": COLLECTION,
                    "dataFilter": {
                        "timeRange": {
                            "from": f"{date_string}T00:00:00Z",
                            "to": f"{date_string}T23:59:59Z"
                        },
                        "mosaickingOrder": "mostRecent"
                    }
                }
            ]
        },
        "output": {
            "width": WIDTH,
            "height": HEIGHT,
            "responses": [
                {
                    "identifier": "default",
                    "format": {
                        "type": "image/tiff"
                    }
                }
            ]
        },
        "evalscript": build_evalscript(used_bands)
    }

    return requests.post(
        f"{BASE_URL}/api/v1/process",
        headers=auth_headers(),
        json=payload,
        timeout=600,
    )

# ============================================================
# 6. DOWNLOAD SELECTED DATES
# ============================================================

downloaded = []
failed = []

for date_string in picked:
    output_path = (
        OUTPUT_DIR / f"ARPS_Para_{date_string}.tif"
    )

    # Do not download an existing non-empty file again
    if (
        output_path.exists()
        and output_path.stat().st_size > 0
    ):
        print(date_string, "- already downloaded, skipping")
        downloaded.append(output_path)
        continue

    try:
        response = process_request(date_string)

        if response.ok:
            output_path.write_bytes(response.content)
            downloaded.append(output_path)

            print(
                date_string,
                "- saved",
                f"{output_path.stat().st_size:,}",
                "bytes",
            )

        else:
            failed.append(date_string)

            print(
                date_string,
                "- FAILED",
                response.status_code,
                response.text[:500],
            )

            # Stop for authentication, permission, or quota errors
            if response.status_code in (401, 403, 429):
                print(
                    "Stopping because of authentication, "
                    "permission, or rate-limit error."
                )
                break

    except Exception as error:
        failed.append(date_string)

        print(
            date_string,
            "- ERROR:",
            repr(error),
        )

        break

    time.sleep(1)

# ============================================================
# 7. SUMMARY
# ============================================================

print()
print("Done")
print("Downloaded:", len(downloaded))
print("Failed:", len(failed))
print("Bands used:", used_bands)
print("Files saved in:", OUTPUT_DIR.resolve())

if failed:
    print("Failed dates:", failed)


Collection: byoc-3f605f75-86c4-411a-b4ae-01c896f0e54e
BBOX: [-52.4825, -6.8735, -52.26725, -6.65775]
BBOX CRS: http://www.opengis.net/def/crs/EPSG/0/4326
Bands: ['blue', 'green', 'red', 'nir']
677 dates found
98 dates selected
Output folder: /Users/huss/data-processing/Github/TechnicalEvaluation-3-planet-dashboard-app-Solutions-Architect/Notebook/planet_downloads
Authentication successful.
2021-01-01 - already downloaded, skipping
2021-01-22 - already downloaded, skipping
2021-02-24 - already downloaded, skipping
2021-03-24 - already downloaded, skipping
2021-04-07 - already downloaded, skipping
2021-04-15 - already downloaded, skipping
2021-05-03 - already downloaded, skipping
2021-05-13 - already downloaded, skipping
2021-05-23 - already downloaded, skipping
2021-05-30 - already downloaded, skipping
2021-06-06 - already downloaded, skipping
2021-06-13 - already downloaded, skipping
2021-06-21 - already downloaded, skipping
2021-06-28 - already downloaded, skipping
2021-07-06 - alread

## 2.11 From per-date files to annual composites

**This step is not contained in any of the three notebooks.** Part 3 loads `composites/composite_2021.tif`, `composite_2022.tif`, `composite_2023.tif` and `composite_All_years.tif` and its error text says "run the composite-generation code once first", but that code is not in the files.
To document the method honestly, a check was run on the data itself: for a block of pixels, the composite for 2021 was compared with the per-pixel **mean** and the per-pixel **median** of the 113 downloaded 2021 files.
The median reproduced the composite closely (mean absolute difference ≈ 5 DN, and many composite values end in `.5`, the typical sign of a median over an even number of scenes), the mean did not (≈ 400-585 DN).
**So the composites are, most likely, per-pixel medians of the per-date files** (an inference: the exact subset of files and the handling of `dataMask` are not recorded).

**Why a median is a sensible choice here:** clouds, cloud shadows and haze are extreme values in a single date; the median over many dates ignores them without needing a cloud mask. The cost is that the composite mixes the whole year, so seasonal effects are averaged in (see Part 3, section 3.9).

**Recommendation:** add the composite-generation code to this part (or to a notebook of its own) so that the workbook is reproducible from the raw downloads. It should read `dataMask` (band 5), set invalid pixels to NaN, take `np.nanmedian` per band over the files of each year (and over all files for "All years") and write 4-band float32 GeoTIFFs (blue, green, red, nir) with the same georeferencing.

## 2.12 What Part 2 gives to Part 3

- `all_images_sentinel.csv`: the catalog of every date available over the AOI (677 items).
- `planet_downloads/ARPS_Para_<date>.tif`: one 5-band float32 GeoTIFF per downloaded date (blue, green, red, nir, dataMask; EPSG:4326; 2400 × 2400 ≈ 10 m).
- The knowledge that annual composites are built from these files (section 2.11).

**Limits of Part 2:** the sandbox covers only 2021-2023; per-date files can be mostly empty when the scene covers only part of the AOI (so some files are only partly filled with valid pixels); downloading consumes Processing Units; and the licence is CC-BY-NC.